# Puxando arquivos

In [ ]:
import pandas as pd
from pathlib import Path
import sys

sys.path.insert(0, str(Path.cwd()))
from sinan_mappings import add_sinan_cbo_labels, standardize_columns

df1 = pd.read_parquet('../data/DENGBR17.parquet')
df2 = pd.read_parquet('../data/DENGBR18.parquet')
df3 = pd.read_parquet('../data/DENGBR19.parquet')

df_todas = pd.concat([df1, df2, df3])
df_todas = standardize_columns(df_todas)

# Separando Colunas para Membro Angel Mansilla

In [ ]:
df = df_todas[['alarm_liver_enlarged', 'infection_country', 'severe_metrorrhagia', 'infection_municipality', 'autoimmune_disease', 'tourniquet_test', 'petechiae_hemorrh', 'tourniquet_test_hemorrh', 'severe_hypotension', 'kidney_disease', 'retro_orbital_pain', 'chik_clinical_form', 'duplicate_flag', 'dengue_hemorrhagic_fever', 'hemorrhagic_evidence', 'joint_pain', 'headache', 'severe_tachycardia', 'alarm_hematocrit_rise', 'symptom_epi_week', 'severe_hematemesis', 'final_classification', 'hematuria', 'viral_isolation_result', 'rash', 'vomiting', 'birth_date', 'severe_weak_pulse', 'race', 'alarm_low_platelets', 'alarm_signs_date', 'severe_bleeding', 'plasma_leakage', 'petechiae', 'pregnancy_status', 'severe_ast_elevated', 'severe_cap_refill', 'severe_myocarditis', 'severe_convulsions']]

# Limpando Colunas

In [ ]:
# Removes columns that do data leakage
cols_leakage = [
    "confirmation_criteria",
    "case_closure_date",

    "alarm_hypotension",
    "alarm_low_platelets",
    "alarm_persistent_vomit",
    "alarm_bleeding",
    "alarm_hematocrit_rise",
    "alarm_abdominal_pain",
    "alarm_lethargy",
    "alarm_liver_enlarged",
    "alarm_fluid_accumul",
    "alarm_signs_date",

    "severe_weak_pulse",
    "severe_convulsions",
    "severe_cap_refill",
    "severe_resp_distress",
    "severe_tachycardia",
    "severe_cold_extremities",
    "severe_hypotension",
    "severe_hematemesis",
    "severe_melena",
    "severe_metrorrhagia",
    "severe_bleeding",
    "severe_ast_elevated",
    "severe_myocarditis",
    "severe_altered_consc",
    "severe_organ_damage",
    "severity_signs_date",
    "infection_country",
    "infection_municipality"
]

df = df.drop(cols_leakage, axis=1, errors='ignore')

In [ ]:
# Removes columns that have all null values or only one unique value
def remove_same_val(df):
    for column in df.columns:
        if df[column].isnull().all():
            print(f"Column '{column}' has all null values.")
            df = df.drop(column, axis=1)
        elif df[column].nunique() == 1:
            print(f"Column '{column}' has only one unique value: {df[column].unique()[0]}")
            df = df.drop(column, axis=1)
    return df

df = remove_same_val(df)

In [ ]:
# Creates columns based on the birth_date
df['birth_date'] = pd.to_datetime(df['birth_date'], errors='coerce')
df['birth_year_derived'] = df['birth_date'].dt.year
df['age'] = 2020 - df['birth_year_derived']

df = df.drop(['birth_date', 'birth_year_derived'], axis=1, errors='ignore')

In [ ]:
# Removes columns related to comorbidities beacuase they don´t have relation to predicting dengue
cols_comorbidities = ['autoimmune_disease', 'kidney_disease']
df = df.drop(cols_comorbidities, axis=1, errors='ignore')

In [ ]:
# Columns that were judged to be not relevant for predicting dengue
cols_irrelevant = ['pregnancy_status', 'race', 'chik_clinical_form', 'viral_isolation_result']
df = df.drop(cols_irrelevant, axis=1, errors='ignore')

In [ ]:
# Separating the symptom_epi_week column into two columns: sem_pri_year and sem_pri_week
def split_sem_pri(df):
    df["symptom_epi_year"] = df["symptom_epi_week"] // 100
    df["symptom_epi_week_number"] = df["symptom_epi_week"] % 100
    df = df.drop(columns=["symptom_epi_week"])
    return df

df = split_sem_pri(df)


In [ ]:
# Mapping the target variable final_classification to binary values
df["final_classification"] = df["final_classification"].map({
    5: 0,    # descartado / não dengue
    10: 1,   # dengue
    11: 1,   # dengue com sinais de alarme
    12: 1    # dengue grave
})

df = df[df["final_classification"].notna()].copy()

df = add_sinan_cbo_labels(df)


# Resultado Final após Colunas Limpas

In [ ]:
print(list(df.columns))
display(df)